In [12]:
import requests
import json
import time
from pathlib import Path
from datetime import datetime
import pandas as pd


In [17]:

BASE_URL = "https://www.tagesschau.de/api2u/news/"
RESSORT = "Ausland"
FROM_DATE = "2025-05-01"
TO_DATE   = "2025-06-01"
PAGE_SIZE = 50             # safe value
MAX_PAGES = 200            # safety stop
SLEEP_SECONDS = 1.5

In [18]:
OUT_DIR = Path("data/experiments/tagesschau_api")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSON = OUT_DIR / "tagesschau_ausland_from_2025-05-01.json"
OUT_CSV  = OUT_DIR / "tagesschau_ausland_from_2025-05-01.csv"

from_dt = datetime.strptime(FROM_DATE, "%Y-%m-%d").date()
to_dt = datetime.strptime(TO_DATE, "%Y-%m-%d").date()


In [19]:
def parse_date(item):
    d = item.get("date")
    if not d:
        return None
    try:
        return datetime.fromisoformat(d.replace("Z", "+00:00")).date()
    except Exception:
        return None


def fetch_page(page):
    params = {
        "ressort": RESSORT,
        "pageSize": PAGE_SIZE,
        "page": page,
    }
    r = requests.get(BASE_URL, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("news", [])


In [20]:
kept_articles = []

print(f"Fetching Tagesschau articles (Ressort={RESSORT}) from {FROM_DATE} onwards...")

for page in range(1, MAX_PAGES + 1):
    news = fetch_page(page)

    if not news:
        print(f"Stopped: no articles on page {page}.")
        break

    dates_on_page = []

    for item in news:
        item_date = parse_date(item)
        if item_date:
            dates_on_page.append(item_date)
            if from_dt <= item_date < to_dt:                    
                kept_articles.append(item)

    print(
        f"Page {page}: fetched {len(news)} | kept {len(kept_articles)} "
        f"| newest={max(dates_on_page)} oldest={min(dates_on_page)}"
    )

    # Stop once we've clearly paged into older content
    if min(dates_on_page) < from_dt:
        print("Reached articles older than May 2025. Stopping.")
        break

    time.sleep(SLEEP_SECONDS)

print(f"\nTotal kept articles: {len(kept_articles)}")

Fetching Tagesschau articles (Ressort=Ausland) from 2025-05-01 onwards...
Page 1: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 2: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 3: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 4: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 5: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 6: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 7: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 8: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 9: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 10: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 11: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 12: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 13: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-04
Page 14: fetched 52 | kept 0 | newest=2026-02-06 oldest=2026-02-

In [24]:
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(kept_articles, f, ensure_ascii=False, indent=2)

df = pd.DataFrame(
    {
        "id": a.get("id"),
        "date": a.get("date"),
        "ressort": a.get("ressort"),
        "title": a.get("title"),
        "firstSentence": a.get("firstSentence"),
        "shareURL": a.get("shareURL"),
    }
    for a in kept_articles
)

df.to_csv(OUT_CSV, index=False)

print(f"Saved JSON → {OUT_JSON}")
print(f"Saved CSV  → {OUT_CSV}")


Saved JSON → data/experiments/tagesschau_api/tagesschau_ausland_from_2025-05-01.json
Saved CSV  → data/experiments/tagesschau_api/tagesschau_ausland_from_2025-05-01.csv


In [25]:
df.head()

""
